# [SK 07 - AI Foundry Agents with Semantic Kernel vs. AI Foundry SDK's](https://github.com/microsoft/semantic-kernel/tree/main/python/samples/getting_started_with_agents/azure_ai_agent)
- How to use Azure AI Agents with Semantic Kernel.
- Dependencies:
  ```
  pip install semantic-kernel azure-ai-projects azure-ai-agents
  ```
- [Sample](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/getting_started_with_agents/azure_ai_agent/step1_azure_ai_agent.py)<br/><br/>

Note: it's worth to review the usage of AI Foundry Agents
- with [Python AI Foundry SDK](https://github.com/maurominella/aaas)
- with [C#](https://github.com/maurominella/aaas/tree/main/FoundryAgents06%20-%20AI%20Foundry%20Agent%20with%20BingGroundingTool)

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

aifoundry_agent_name = "aifoundry_agent-PYTHON"
sk_agent_name = "sk_agent-PYTHON"

instructions  = "you are a clever agent"
user_inputs = [
    "Toggle the status of my second light.",
    "Toggle the status of the third light.",
    "Retrieve the status of all lights."
]

plugin_name                 = "Lights"

project_endpoint = os.environ["PROJECT_ENDPOINT"]
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"]
credential = DefaultAzureCredential()

print(f'Project Endpoint: <{project_endpoint}>')
print(f'OpenAI API Version: <{openai_api_version}>')
print(f"azure-ai-projects library installed version: <{importlib.metadata.version("azure-ai-projects")}>")
print(f"azure-ai-agents library installed version: <{importlib.metadata.version("azure-ai-agents")}>")

Environment variables have been loaded ;-)
Project Endpoint: <https://aiservicesiyva.services.ai.azure.com/api/projects/newstrdproject01iyva>
OpenAI API Version: <2025-01-01-preview>
azure-ai-projects library installed version: <1.0.0b11>
azure-ai-agents library installed version: <1.1.0b1>


# Native Plugin

In [2]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
    
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# CREATE AI FOUNDRY PROJECT CLIENT...

## ...using AI Foundry SDK: `aifoundry_project_client`
Please recall that `project_client.agents` == agents_client

In [3]:
from azure.ai.projects import AIProjectClient

aifoundry_project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# List current AI Foundry Agents using AI Foundry SDK
agents_by_aifoundry = aifoundry_project_client.agents.list_agents()
display([f"{a.id}: {a.name}" for a in agents_by_aifoundry])

aifoundry_project_client

['asst_eMhLrHS9ACY8vEIvt3MoMIq9: aifoundry_agent-PYTHON',
 'asst_6mv9Kq8ko38WYkLYQNUMfj2s: aifoundry_agent-PYTHON',
 'asst_vrFydr2vU7rcKzhZMDeLRM91: aifoundry_agent-PYTHON',
 'asst_jL8MxQT1N0T3UbY541cEEQDN: aifoundry_agent-PYTHON',
 'asst_fNqG2oapAZm5urPQdEhHv2gm: aifoundry_agent-PYTHON',
 'asst_sx0ap2KNmWhjtINqhc4ZhJHz: aifoundry_agent-PYTHON',
 'asst_K1SUxbPpmQTmDYrCdibgk9br: aifoundry_agent-PYTHON',
 'asst_dCzSigdAn3w86gjWKL9bR2lM: aifoundry_agent-PYTHON',
 'asst_UFEg13LqJpOuD1Wjk89cZ7Oh: aifoundry_agent-PYTHON',
 'asst_2T9qqdoxF9aPoPSyicSodMZz: aifoundry_agent-PYTHON',
 'asst_dxnZJ1743jOvfqvt0mE4pvLA: aifoundry_agent-PYTHON',
 'asst_eKXE7viUxGyBztOu8ebRH7JL: aifoundry_agent-PYTHON']

## ...using Semantic Kernel SDK: `sk_project_client`

In [4]:
os.environ["AZURE_AI_AGENT_ENDPOINT"] = os.environ["AZURE_OPENAI_ENDPOINT"]
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] = os.environ["MODEL_DEPLOYMENT_NAME"]
os.environ["AZURE_AI_AGENT_API_VERSION"] = os.environ["AZURE_OPENAI_API_VERSION"]

from semantic_kernel.agents import AzureAIAgent

sk_project_client = AzureAIAgent.create_client(
    credential=DefaultAzureCredential(),
    api_version=openai_api_version,
)

sk_project_client

# CREATE AI FOUNDRY AGENT

## ...using AI Foundry SDK: `aifoundry_ai_agent`
Single step:
- `create_agent` for agent **creation**

In [5]:
# using project_client.agents...
aifoundry_ai_agent = aifoundry_project_client.agents.create_agent(
    model=deployment_name,
    name=aifoundry_agent_name,
    instructions=instructions,
    # tools=code_interpreter.definitions,
    # tool_resources=code_interpreter.resources,
)

print(f"Created agent, agent ID: {aifoundry_ai_agent.id},\nagent items: {aifoundry_ai_agent.items}")

Created agent, agent ID: asst_DJNlQtlbFMdgTwGniOB0B1T6,
agent items: <bound method _MyMutableMapping.items of {'id': 'asst_DJNlQtlbFMdgTwGniOB0B1T6', 'object': 'assistant', 'created_at': 1749158799, 'name': 'aifoundry_agent-PYTHON', 'description': None, 'model': 'gpt-4o', 'instructions': 'you are a clever agent', 'tools': [], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}>


## ...using Semantic Kernel SDK: `aifoundry_ai_agent`
Two steps:
- use `create_agent` for **agent definition**
- use `AzureAIAgent` for **client creation** (including **kernel**)

Possible bug: Make sure that the constant value for DEFAULT_AZURE_API_VERSION in envs\<semantic_kernel_env_name>\Lib\site-packages\semantic_kernel\connectors\ai\open_ai\const.py matches your OPENAI_API_VERSION environment value (listed for example in credentials_my.env).

In [6]:
# use create_agent for agent definition
sk_ai_agent_definition = await sk_project_client.agents.create_agent(
    model=deployment_name,
    name=sk_agent_name,
    instructions=instructions,
)

sk_ai_agent_definition

ResourceNotFoundError: (404) Resource not found
Code: 404
Message: Resource not found

In [ ]:
# use create_agent for agent definition

sk_ai_agent_definition = await sk_project_client.agents.create_agent(
    model=model_deployment_name,
    name=f"{agent_name}_SK",
    instructions=instructions
)
sk_ai_agent_definition

In [ ]:
# use AzureAIAgent for client creation (including kernel)

sk_ai_agent = AzureAIAgent(
    client=sk_project_client,
    definition=sk_ai_agent_definition,
)

sk_ai_agent

# ADD PLUGIN TO THE AGENT

## AI Foundry SDK

In [ ]:
from azure.ai.projects.models import FunctionTool, ToolSet
from typing import Any, Callable, Set

# Instantiate the LightsPlugin class
lights_plugin = LightsPlugin()

# Create a set of the desired callable methods
plugin_functions: Set[Callable[..., Any]] = {
    lights_plugin.get_state,
    lights_plugin.change_state,
}

# Pass these functions to FunctionTool
functions = FunctionTool(plugin_functions)

# Add the plugin to the agent
aifoundry_project_client.agents.update_agent(tools=functions.definitions, agent_id=aifoundry_ai_agent.id)

aifoundry_ai_agent.items

## Semantic Kernel SDK

In [ ]:
sk_ai_agent.kernel.add_plugin(
    plugin=LightsPlugin(),
    plugin_name=plugin_name,
)
sk_ai_agent

# CREATE A NEW THREAD (EVEN AN EMPTY ONE)

## AI Foundry SDK

In [ ]:
aifoundry_thread = aifoundry_project_client.agents.create_thread()
print(f"Created thread: {aifoundry_thread}\n")

## Semantic Kernel SDK
If no thread is provided, a new thread will be created and returned with the initial response

In [ ]:
from semantic_kernel.agents import AzureAIAgentThread

thread: AzureAIAgentThread = None

# MESSAGE(S) CREATION

## AI Foundry SDK

In [ ]:
for user_input in user_inputs:
    message = aifoundry_project_client.agents.create_message(
        thread_id=aifoundry_thread.id, 
        role="user", 
        content=user_input,
    )
    print(f"Created message: {message}")

## Semantic Kernel SDK

In [ ]:
# user messages do not need to be packaged into a message, since they can be passed as simple strings

# RUN THE AGENT

## AI Foundry SDK

In [ ]:
%%time
import time

print(f"Running the agent {aifoundry_thread.id} on the thread {aifoundry_ai_agent.id}")

run = aifoundry_project_client.agents.create_run(thread_id=aifoundry_thread.id, agent_id=aifoundry_ai_agent.id)

while run.status in ['queued', 'in_progress', 'cancelling']:
    time.sleep(1)
    run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
    print(f"Run status: {run.status}")

print(f"Run finished with status: {run.status}.\n\nRun: {run}")

if run.status == "failed":
    # Check if you got "Rate limit is exceeded.", then you want to get more quota
    print(f"Run failed: {run.last_error}")

In [ ]:
# TO BE FIXED FOR AI FOUNDRY

In [ ]:
# KEEP RUNNING UNTIL RunStatus.COMPLETED

# just for checking: analyze the current status
import time, json
from azure.ai.projects.models import FunctionTool, ToolOutput

run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
print(f"Initial run status: {run.status}")
print(f"\nRequired action(s): {run.required_action}")
print(f"\nWe need to run {len(run.required_action.submit_tool_outputs.tool_calls)} tool call(s): {run.required_action.submit_tool_outputs.tool_calls}")

i = 0
tool_outputs = []
for tool_call in run.required_action.submit_tool_outputs.tool_calls:
    i += 1
    output = functions.execute(tool_call)
    print(f"output: {output}")
    output=json.dumps(output) # TRYING TO PATCH, BUT IT STILL DOES NOT WORK
    
    print(f"\n{i} - Executing tool_call {tool_call.function.name} ({tool_call.id}) >>> output: {output}")
    tool_outputs.append(
        ToolOutput(
            tool_call_id=tool_call.id,
            output=output
        )
    )
    
run = aifoundry_project_client.agents.submit_tool_outputs_to_run(
    thread_id=aifoundry_thread.id, run_id=run.id, tool_outputs=tool_outputs
)

while run.status in ["queued", "in_progress"]:
    time.sleep(1)
    run = aifoundry_project_client.agents.get_run(thread_id=aifoundry_thread.id, run_id=run.id)
    print(f"\nFinal run status: {run.status}")

## Semantic Kernel SDK

In [ ]:
for user_input in user_inputs:
    print(f"# User: {user_input}")
    # Invoke the agent with the specified message for response
    response = await sk_ai_agent.get_response(messages=user_input, thread=thread) # or sk_ai_agent.invoke(messages=user_input, thread=thread)
    thread = response.thread
    print(f"--> # {response.name}: {response}\n")

# Fetch messages from the thread after the agent run execution

## AI Foundry SDK

In [ ]:
from azure.ai.projects.models import MessageTextContent, MessageImageFileContent

if run.status == 'completed':    
    messages = aifoundry_project_client.agents.list_messages(thread_id=aifoundry_thread.id)
    messages_nr = len(messages.data)
    print(f"Here are the {messages_nr} messages:\n")
    
    for i, message in enumerate(reversed(messages.data), 1):
        j = 0
        print(f"\n===== MESSAGE {i} =====")
        for c in message.content:
            j +=1
            if (type(c) is MessageImageFileContent):
                print(f"\nCONTENT {j} (MessageImageFileContent) --> image_file id: {c.image_file.file_id}")
            elif (type(c) is MessageTextContent):
                print(f"\nCONTENT {j} (MessageTextContent) --> Text: {c.text.value}")
                for a in c.text.annotations:
                    print(f">>> Annotation in MessageTextContent {j} of message {i}: {a.text}\n")

else:
    print(f"Sorry, I can't proceed because the run status is {run.status}")

## Semantic Kernel SDK

In [ ]:
# Get ready to print all messages of a AzureAIAgentThread object

async def print_messages(thread: AzureAIAgentThread):
    i=0
    messages = [message async for message in thread.get_messages()] # this doesn't work: messages = await thread.get_messages()
    for cmc in messages: # ChatMessageContent
        if cmc.inner_content is None:
            i += 1
            if cmc.role.value == "user" or cmc.role.value == "assistant":
                print(f"{i} - Role: {cmc.role.value}, text: {cmc.items[0].text}")
            elif cmc.role.value == "tool":
                print(f"{i} - Role: {cmc.role.value}, function_name: {cmc.items[0].name}")
        else:
            for choice in cmc.inner_content.choices:
                if choice.message.tool_calls is None:
                    i += 1
                    print(f"{i} - Finish reason: {choice.finish_reason}")
                else:
                    for tc in choice.message.tool_calls:
                        i += 1
                        print (f"{i} - Function call: {tc.function.name}({tc.function.arguments})")
    return messages

messages = await print_messages(thread)

# HIC SUNT LEONES

# TEARDOWN

## AI Foundry SDK

In [ ]:
# delete thread

print(f"Deleting thread {aifoundry_thread}...")
aifoundry_project_client.agents.delete_thread(aifoundry_thread.id)

In [ ]:
# delete agent
aifoundry_project_client.agents.delete_agent(aifoundry_ai_agent.id)

# Semantic Kernel SDK

In [ ]:
# delete thread

print(f"Deleting thread {thread.id}...")
await thread.delete()

In [ ]:
# delete agent
await sk_project_client.agents.delete_agent(sk_ai_agent.id)

# HIC SUNT LEONES

In [ ]:
# delete all agents
agents_list = aifoundry_project_client.agents.list_agents(limit=100).data # max limit is 100
print(f"There are {len(agents_list)} agents to delete")

i = 0
for agent in agents_list:
    i += 1
    print(f"Agent {i}/{len(agents_list)}: Agent {agent.name} ({agent.id})) is being deleted...")
    aifoundry_project_client.agents.delete_agent(agent.id) # comment / un-comment this line if you want to delete it